In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers tokenizers
%pip install torch==2.4.1 torchvision==0.19.1 transformers==4.44.2 tqdm pandas --no-cache-dir

In [2]:
import os
import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.0.1
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [8]:
if torch.cuda.is_available():
    print(f"✅ GPU aktywne: {torch.cuda.get_device_name(0)}")
    device = "cuda"
elif torch.backends.mps.is_available():
    print("✅ MPS aktywne (Mac)")
    device = "mps"
else:
    print("⚠️ Działa tylko CPU. Sprawdź ustawienia instancji.")
    device = "cpu"

⚠️ Działa tylko CPU. Sprawdź ustawienia instancji.


In [8]:
classifier = pipeline(
    "text-classification",
    model="monologg/bert-base-cased-goemotions-original",
    top_k=None,
    device=device,
    token=os.getenv("HF_TOKEN"),
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
df = pd.read_csv("../../../data/cleaned_data.csv")

print("Rozpoczynam klasyfikację...")
texts = df['cleaned_statement'].astype(str).tolist()

Rozpoczynam klasyfikację...


In [ ]:
results = []
for out in tqdm(classifier(texts, batch_size=16, truncation=True), total=len(texts)):
    results.append(out)

In [ ]:
extracted_data = []
for res in results:
    row = {item['label']: item['score'] for item in res}
    extracted_data.append(row)

In [ ]:
features_df = pd.DataFrame(extracted_data)

In [ ]:
if 'label' in df.columns:
    features_df['target_label'] = df['label'].values

features_df.to_csv('../../../data/features_for_model.csv', index=False)